# BERT (enunciados 1-3 e 6J)

### 1. Configure um ambiente no Google Colab com TensorFlow, Keras e keras_hub. Carregue um modelo BERT (bert_base_en_uncased) e seu respectivo pré-processador. Explore resumidamente a arquitetura e identifique os componentes responsáveis pela geração das representações de entrada.

In [1]:
import tensorflow as tf
import keras_hub

I0000 00:00:1795185797.627668   31449 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1795185797.663245   31449 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1795185798.594853   31449 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
bert = keras_hub.models.BertBackbone.from_preset("bert_base_en_uncased")
preprocessor = keras_hub.models.BertTextClassifierPreprocessor.from_preset("bert_base_en_uncased", sequence_length=16)

W0000 00:00:1795185800.425544   31449 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


O modelo e o pré-processador foram carregados, agora vamos explorar a arquitetura do `bert_base_en_uncased`

In [3]:
config = bert.get_config()

In [4]:
for label, key in [
    ("# blocos do encoder", "num_layers"),
    ("# cabeças de self-attention por bloco", "num_heads"),
    ("Dimensão oculta", "hidden_dim"),
    ("Dimensão intermediária da FFN", "intermediate_dim"),
    ("Comprimento máximo de uma sequência", "max_sequence_length"),
]:
    print(f"{label}: {config[key]}")

# blocos do encoder: 12
# cabeças de self-attention por bloco: 12
Dimensão oculta: 768
Dimensão intermediária da FFN: 3072
Comprimento máximo de uma sequência: 512


O `bert_base_en_uncased` tem **12 blocos**, cada bloco tem **12 heads de self-attention** e uma rede feed-forward com dimensão intermediária **3072**. A dimensão oculta é **768**, ou seja, cada posição é representada por um vetor de 768 valores na entrada e na saída de cada bloco.

Pela [documentação do pré-processador](https://keras.io/keras_hub/api/models/bert/bert_text_classifier_preprocessor/), o pré-processador separa o texto em tokens WordPiece, acrescenta os tokens especiais `[CLS]` e `[SEP]`, completa a sequência com `[PAD]` quando necessário e devolve `token_ids`, `segment_ids` e `padding_mask`. Aqui, `sequence_length=16` fixa o comprimento da saída do pré-processador. A tabela posicional do modelo comporta **512 posições**, de 0 a 511.

In [5]:
print("Tabelas de embeddings (# linhas, dimensão do vetor)\n")
for name in ("token_embedding", "position_embedding", "segment_embedding"):
    layer = bert.get_layer(name)
    print(f"{name}: {tuple(layer.weights[0].shape)}")

Tabelas de embeddings (# linhas, dimensão do vetor)

token_embedding: (30522, 768)
position_embedding: (512, 768)
segment_embedding: (2, 768)


A camada `token_embedding` busca um vetor de 768 valores pelo ID do token. A tabela tem 30.522 linhas, uma para cada token do vocabulário.

O `position_embedding` usa o índice absoluto do token na sequência, começando em zero. O preset configura `max_sequence_length=512`, então a tabela tem uma linha aprendida para cada posição de 0 a 511. Com `sequence_length=16` no pré-processador, o BERT usa as linhas de 0 a 15 para montar os vetores de entrada. Esse comprimento inclui os tokens especiais e, quando necessário, o preenchimento. No pré-treinamento original, o BERT usou sequências de comprimento 512 nos 10% finais dos passos, quando aprendeu os embeddings das posições mais longas.

O `segment_embedding` informa a qual trecho cada token pertence quando o BERT recebe um par como `[CLS] A [SEP] B [SEP]`. O segmento A usa o ID 0 e o segmento B usa o ID 1. Essa informação foi usada no pré-treinamento original do BERT para a tarefa de *Next Sentence Prediction* (NSP). Em metade dos exemplos, B era o trecho que vinha logo depois de A no corpus. Na outra metade, B era um trecho sorteado. O ID 1 indicava que o token pertencia a B em ambos os casos. O modelo usava a representação contextualizada de `[CLS]` para prever se B era a próxima sentença ou não. Neste trabalho, cada frase forma uma entrada independente, então o BERT soma o mesmo vetor de segmento 0 aos tokens válidos de ambas.

In [6]:
for name in ("embeddings_add", "embeddings_layer_norm", "embeddings_dropout"):
    layer = bert.get_layer(name)
    print(f"{layer.name} ({type(layer).__name__}): {tuple(layer.output.shape)}")

print(f"\nEixo da normalização: {bert.get_layer('embeddings_layer_norm').axis}")
print(f"Taxa de dropout: {bert.get_layer('embeddings_dropout').rate:.0%}")

embeddings_add (Add): (None, None, 768)
embeddings_layer_norm (LayerNormalization): (None, None, 768)
embeddings_dropout (Dropout): (None, None, 768)

Eixo da normalização: [-1]
Taxa de dropout: 10%


Então, para um token de ID $t$ na posição $p$ e no segmento $s$, a camada `embeddings_add` soma os três vetores elemento a elemento.

$$
\mathbf{x}_p = \mathbf{E}_{\mathrm{token}}[t] + \mathbf{E}_{\mathrm{pos}}[p] + \mathbf{E}_{\mathrm{seg}}[s]
$$

A camada `embeddings_layer_norm` normaliza a soma usando a média e a variância dos 768 valores de cada posição. Em seguida, `embeddings_dropout` zera aleatoriamente componentes do vetor em modo de treinamento, com probabilidade de 10% nesse preset. Em inferência, o dropout fica desativado. O vetor resultante alimenta o primeiro bloco Transformer.


In [7]:
mask_input = bert.get_layer("padding_mask")
print(f"\n{mask_input.name} ({type(mask_input).__name__}): {tuple(mask_input.output.shape)}")

mask_example = preprocessor(tf.constant(["Hello world!"]))["padding_mask"]
print("Máscara para 'Hello world!':", mask_example[0].numpy())


padding_mask (InputLayer): (None, None)
Máscara para 'Hello world!': [ True  True  True  True  True False False False False False False False
 False False False False]


A entrada `padding_mask` marca com `True` as posições válidas e com `False` as posições inseridas para completar a sequência. O BERT passa essa máscara em cada bloco Transformer. Assim, no self-attention, os tokens válidos podem usar outros tokens válidos como contexto e ignoram as posições de preenchimento.

---

### 2. Utilize o modelo para processar as sentenças abaixo:

- “The developer wrote the code with the new keyboard.”;
- “The developer wrote the code with the new mouse.”
  
### Apresente os tokens produzidos pelo pré-processador e obtenha as representações de entrada do BERT. Identifique os token embeddings e as representações posicionais aprendidas (position embeddings). Informe as dimensões dos tensores e explique brevemente como essas representações são combinadas.

In [8]:
sentences = tf.constant([
    "The developer wrote the code with the new keyboard.",
    "The developer wrote the code with the new mouse.",
])
inputs = preprocessor(sentences)

tokenizer = preprocessor.tokenizer
token_ids = inputs["token_ids"].numpy().tolist()
tokens = [[tokenizer.id_to_token(token_id) for token_id in sentence_ids] for sentence_ids in token_ids]

print(f"{'Posição':>7} | {'Token (frase 1)':<16} {'ID':>5} | {'Token (frase 2)':<16} {'ID':>5}")
for position, (first, second) in enumerate(zip(tokens[0], tokens[1])):
    print(f"{position:>7} | {first:<16} {token_ids[0][position]:>5} | {second:<16} {token_ids[1][position]:>5}")

print("\n\n")
for name, tensor in inputs.items():
    print(f"{name}: {tuple(tensor.shape)}")

Posição | Token (frase 1)     ID | Token (frase 2)     ID
      0 | [CLS]              101 | [CLS]              101
      1 | the               1996 | the               1996
      2 | developer         9722 | developer         9722
      3 | wrote             2626 | wrote             2626
      4 | the               1996 | the               1996
      5 | code              3642 | code              3642
      6 | with              2007 | with              2007
      7 | the               1996 | the               1996
      8 | new               2047 | new               2047
      9 | keyboard          9019 | mouse             8000
     10 | .                 1012 | .                 1012
     11 | [SEP]              102 | [SEP]              102
     12 | [PAD]                0 | [PAD]                0
     13 | [PAD]                0 | [PAD]                0
     14 | [PAD]                0 | [PAD]                0
     15 | [PAD]                0 | [PAD]                0



token_ids: 

In [9]:
from keras import Model

embedding_layers = ("token_embedding", "position_embedding", "segment_embedding",
                    "embeddings_add", "embeddings_layer_norm", "embeddings_dropout")

embedding_model = Model(
    inputs={name: bert.input[name] for name in ("token_ids", "segment_ids")},
    outputs={name: bert.get_layer(name).output for name in embedding_layers},
)

embedding_inputs = {name: inputs[name] for name in ("token_ids", "segment_ids")}
embeddings = embedding_model(embedding_inputs, training=False)
token_embeddings = embeddings["token_embedding"]
position_embeddings = embeddings["position_embedding"]
encoder_inputs = embeddings["embeddings_dropout"]

for name, tensor in embeddings.items():
    print(f"{name}: {tuple(tensor.shape)}")

token_embedding: (2, 16, 768)
position_embedding: (2, 16, 768)
segment_embedding: (2, 16, 768)
embeddings_add: (2, 16, 768)
embeddings_layer_norm: (2, 16, 768)
embeddings_dropout: (2, 16, 768)


In [10]:
position = tokens[0].index("keyboard")
print(f"Posição {position}: na frase 1: {tokens[0][position]}, na frase 2: {tokens[1][position]}.")

for label, vectors in [("Embeddings de token", token_embeddings), ("Embeddings posicionais", position_embeddings)]:
    equal = tf.reduce_all(vectors[0, position] == vectors[1, position])
    print(f"{label} iguais nessa posição? {bool(equal.numpy())}")

Posição 9: na frase 1: keyboard, na frase 2: mouse.
Embeddings de token iguais nessa posição? False
Embeddings posicionais iguais nessa posição? True


As duas frases têm 12 posições válidas, contando `[CLS]`, o ponto final e `[SEP]`. O pré-processador acrescenta 4 `[PAD]` para completar as 16 posições. Cada tensor do pré-processador tem formato `(2, 16)`. Cada tensor de embeddings exibido tem formato `(2, 16, 768)`, com os eixos sendo **frase, posição e componente do vetor**.

Na posição **9**, `keyboard` e `mouse` têm IDs diferentes e consultam vetores diferentes na tabela de tokens. As duas usam a **mesma linha da tabela posicional aprendida** porque ocupam o mesmo índice. Como as frases são entradas independentes, ambas também usam o vetor de segmento 0 nessa posição.

O BERT soma os vetores de token, posição e segmento elemento a elemento em `embeddings_add`. Após a normalização, a saída de `embeddings_dropout` é a representação que entra no primeiro bloco Transformer.

---

### 3. Visualize, por meio de um mapa de calor (heatmap), as representações posicionais aprendidas pelo BERT para as primeiras 30 posições. Com base na visualização, descreva brevemente como os valores variam entre posições e dimensões. Não é necessário realizar uma análise quantitativa extensa.